# 🧠 从零开始写神经网络

> 面向新手：只用 `numpy` 和 `matplotlib`，不调用任何深度学习框架，把一个能解 **XOR** 的多层感知机从零敲出来。

## 🎯 学完这篇你会得到什么

1. 知道「感知机」到底是什么 —— 一个会学习的二分类小函数
2. 亲手用几十行代码写出一个能学 AND / OR 的感知机
3. 看到感知机为什么会输给 **XOR**
4. 亲手实现一个两层感知机（MLP），并手写反向传播
5. 训练它把 XOR 这种「线性不可分」的问题解决

## 🗺️ 路线图

| 步骤 | 内容 | 你会写什么 |
|---|---|---|
| 1 | 感知机是什么 | 一句话定义 + 画图解释 |
| 2 | 最小感知机 forward | 10 行代码算 `y = step(w·x + b)` |
| 3 | 学习规则 | 用「错一次就改一点」的方式调权重 |
| 4 | 训练 AND / OR | 画训练曲线 + 决策边界 |
| 5 | XOR 翻车 | 演示感知机的根本缺陷 |
| 6 | 多层感知机 | 加一个隐藏层 + sigmoid |
| 7 | 反向传播 | 用链式法则手算梯度 |
| 8 | 解决 XOR | 看到神经网络第一次「开窍」 |

> 💡 建议：每个 cell 跑一下再往下看。改改数字、画画图，比光看有用得多。

## 🧩 1. 感知机（Perceptron）是什么？

把它想成一个**会打分的迷你决策器**：

```
输入:  x1, x2, ..., xn        (特征)
       ↓
权重:  w1, w2, ..., wn        (每个特征的重要性)
偏置:  b                       (门槛)
       ↓
求和:  z = w1·x1 + w2·x2 + ... + wn·xn + b
       ↓
激活:  y = step(z)             (≥ 0 输出 1，否则 0)
输出:  0 或 1                   (二分类)
```

### 几何上它在干什么？

- `z = w·x + b = 0` 在平面上是一条**直线**（高维里是一个超平面）
- 这条直线把平面分成两半，一半预测 1，一半预测 0
- 所以**单层感知机只能切一条直线**

### 关键直觉

> 感知机 = 一次线性打分 + 一次硬阈值。
> 「学习」= 找到合适的 `w` 和 `b`，让那条直线把正负样本分对。

下面就动手写。

### 🧪 2. 最小感知机：手算 AND

下面是一个**写死**权重的感知机（`w=[1,1], b=-1.5`），它刚好能算 AND：

- 只有 (1,1) 时 `1+1-1.5=0.5>0` → 输出 1
- 其他三种 `z<0` → 输出 0

In [15]:
import numpy as np

# ---------- 最小感知机：只有 forward ----------
def step(z):
    """阶跃激活函数：>=0 返 1，否则 0"""
    return (z >= 0).astype(int)

def perceptron_forward(x, w, b):
    """
    x: shape (n, ) 一个样本
    w: shape (n, ) 权重
    b: 标量 偏置
    返回: 0 或 1
    """
    z = np.dot(x, w) + b   # 加权求和
    return step(z)         # 过阶跃

# ---------- 写死一组"刚好能算 AND"的权重 ----------
w = np.array([1.0, 1.0])
b = -1.5

# AND 真值表
X = np.array([[0,0],
              [0,1],
              [1,0],
              [1,1]])
y_true = np.array([0, 0, 0, 1])

# 跑一遍
print("x1 x2 | z       | y_pred")
print("-" * 30)
for xi, yi in zip(X, y_true):
    z = np.dot(xi, w) + b
    y_pred = perceptron_forward(xi, w, b)
    print(f"{xi[0]}  {xi[1]}  | {z:+.2f}  |  {y_pred}   (真值={yi})")

x1 x2 | z       | y_pred
------------------------------
0  0  | -1.50  |  0   (真值=0)
0  1  | -0.50  |  0   (真值=0)
1  0  | -0.50  |  0   (真值=0)
1  1  | +0.50  |  1   (真值=1)


### 📖 3. 感知机怎么「学」？

刚才的 `w` 和 `b` 是我手动凑出来的。能不能让机器**自己找**？

感知机学习规则（Rosenblatt, 1958）极其朴素：

```
对每个样本 (x, y_true):
    y_pred = step(w·x + b)
    err    = y_true - y_pred        # 错得越多，err 越大（-1, 0, 1）
    w     += lr * err * x           # 按 err 修正权重
    b     += lr * err               # 修正偏置
```

> 直觉：**预测低了就把权重往正方向推，预测高了就往负方向推。**
> 这个规则只有在数据**线性可分**时才会收敛（Minsky & Papert 已经证明）。

下面动手写一个会自己学的感知机。

### 💻 4. 一个会自己学的感知机

把感知机包成一个类，让它能：
- 接收一批数据
- 反复跑 forward → 算错 → 改权重
- 记录每轮错误数，画训练曲线

In [14]:
class Perceptron:
    """
    最小感知机：单层、二分类、阶跃激活、感知机学习规则
    """
    def __init__(self, n_features, lr=0.1, seed=0):
        rng = np.random.default_rng(seed)
        # 随机初始化权重（小一点的随机数）
        self.w = rng.normal(loc=0.0, scale=0.1, size=n_features)
        self.b = 0.0
        self.lr = lr
        self.errors_history = []   # 记录每一轮的错分类数

    def forward(self, x):
        """对一个样本做前向，返回 0/1"""
        z = np.dot(x, self.w) + self.b
        return int(z >= 0)

    def predict(self, X):
        """对一批样本做预测"""
        z = X @ self.w + self.b
        return (z >= 0).astype(int)

    def fit(self, X, y, epochs=20, verbose=True):
        """
        训练
        X: (N, n_features)
        y: (N,)  值是 0/1
        """
        for epoch in range(1, epochs + 1):
            errors = 0
            # 随机打乱顺序，训练更稳
            idx = np.random.permutation(len(X))
            for i in idx:
                xi, yi = X[i], y[i]
                y_pred = self.forward(xi)
                err = yi - y_pred              # -1, 0, 1
                if err != 0:
                    errors += 1
                self.w += self.lr * err * xi  # 权重更新
                self.b += self.lr * err        # 偏置更新
            self.errors_history.append(errors)
            if verbose:
                print(f"epoch {epoch:2d}  errors={errors}  w={self.w}  b={self.b:.3f}")
        return self


# ---------- 用它学 AND ----------
X = np.array([[0,0],[0,1],[1,0],[1,1]])
y_and = np.array([0, 0, 0, 1])

ppn = Perceptron(n_features=2, lr=0.1, seed=42)
ppn.fit(X, y_and, epochs=10)

print("\n学完之后预测：")
print("y_pred:", ppn.predict(X))
print("y_true:", y_and)

epoch  1  errors=3  w=[ 0.03047171 -0.00399841]  b=-0.100
epoch  2  errors=2  w=[0.03047171 0.09600159]  b=-0.100
epoch  3  errors=0  w=[0.03047171 0.09600159]  b=-0.100
epoch  4  errors=0  w=[0.03047171 0.09600159]  b=-0.100
epoch  5  errors=0  w=[0.03047171 0.09600159]  b=-0.100
epoch  6  errors=0  w=[0.03047171 0.09600159]  b=-0.100
epoch  7  errors=0  w=[0.03047171 0.09600159]  b=-0.100
epoch  8  errors=0  w=[0.03047171 0.09600159]  b=-0.100
epoch  9  errors=0  w=[0.03047171 0.09600159]  b=-0.100
epoch 10  errors=0  w=[0.03047171 0.09600159]  b=-0.100

学完之后预测：
y_pred: [0 0 0 1]
y_true: [0 0 0 1]


### 🧪 4.5 砍掉一个参数会怎样？

新手常问：「`w` 和 `b` 是不是都必需？只要一个不行吗？」

直接做实验对比：
1. **只有 `w`**：`z = w·x`，`b` 锁死为 0
2. **只有 `b`**：`z = b`，`w` 锁死为 0
3. **完整版**：`z = w·x + b`

In [16]:
def fit_variant(X, y, learn_w=True, learn_b=True, epochs=20, lr=0.1, seed=0):
    """感知机的可配置版本：可以关掉 w 或 b 的学习"""
    rng = np.random.default_rng(seed)
    w = rng.normal(0, 0.1, 2) if learn_w else np.zeros(2)
    b = 0.0
    history = []   # 每轮的 (w, b, errors)
    for ep in range(epochs):
        errors = 0
        for xi, yi in zip(X, y):
            z = np.dot(xi, w) + b
            y_pred = int(z >= 0)
            err = yi - y_pred
            if err != 0: errors += 1
            if learn_w: w += lr * err * xi   # 可关
            if learn_b: b += lr * err         # 可关
        history.append((w.copy(), b, errors))
    return w, b, history

# ---------- 实验 ----------
X = np.array([[0,0],[0,1],[1,0],[1,1]])
y_and = np.array([0, 0, 0, 1])

def step_scalar(z): return int(z >= 0)

print("=" * 55)
print("【只有 w】learn_w=True, learn_b=False")
print("=" * 55)
w, b, _ = fit_variant(X, y_and, learn_w=True, learn_b=False, epochs=20)
preds = [step_scalar(np.dot(xi, w) + b) for xi in X]
print(f"最终: w={w.round(3)}, b={b:.3f}, 预测={preds}, 真值={y_and.tolist()}")
print("→ (0,0) 永远被预测成 1（因为 z=0 触发 step）\n")

print("=" * 55)
print("【只有 b】learn_w=False, learn_b=True")
print("=" * 55)
w, b, _ = fit_variant(X, y_and, learn_w=False, learn_b=True, epochs=20)
preds = [step_scalar(np.dot(xi, w) + b) for xi in X]
print(f"最终: w={w.round(3)}, b={b:.3f}, 预测={preds}, 真值={y_and.tolist()}")
print("→ 4 个样本预测都一样（常数分类器），AND 永远至少错 1 个\n")

print("=" * 55)
print("【w + b 完整】learn_w=True, learn_b=True")
print("=" * 55)
w, b, _ = fit_variant(X, y_and, learn_w=True, learn_b=True, epochs=20)
preds = [step_scalar(np.dot(xi, w) + b) for xi in X]
print(f"最终: w={w.round(3)}, b={b:.3f}, 预测={preds}, 真值={y_and.tolist()}")
print("→ ✅ 完美解开 AND")

【只有 w】learn_w=True, learn_b=False
最终: w=[0.013 0.087], b=0.000, 预测=[1, 1, 1, 1], 真值=[0, 0, 0, 1]
→ (0,0) 永远被预测成 1（因为 z=0 触发 step）

【只有 b】learn_w=False, learn_b=True
最终: w=[0. 0.], b=0.000, 预测=[1, 1, 1, 1], 真值=[0, 0, 0, 1]
→ 4 个样本预测都一样（常数分类器），AND 永远至少错 1 个

【w + b 完整】learn_w=True, learn_b=True
最终: w=[0.213 0.187], b=-0.300, 预测=[0, 0, 0, 1], 真值=[0, 0, 0, 1]
→ ✅ 完美解开 AND


### 💡 结论

- **`w` 的作用**：控制决策边界的**方向**（直线的斜率）
- **`b` 的作用**：控制决策边界的**位置**（直线离原点多远）
- 没有 `b` → 直线**过原点**，卡在原点上动不了
- 没有 `w` → 没有方向，所有样本共用同一条"等高线"
- 两者配合，直线才能在平面上**任意摆**

> 🤓 一个记忆口诀：
> **「`w` 决定直线的角度，`b` 决定直线的位置」**
> 角度不对切错，位置不对也切错。

现实里的神经网络（包括 MLP、CNN、Transformer）**每一层都同时有 `W` 和 `b`**。砍掉 `b` 等于强迫所有"分类直线"都穿过原点——表达能力会大打折扣。

### 📊 5. 训练曲线 + 决策边界

两个图能让你一眼看明白感知机在干什么：
- **左图**：每轮错分类数随 epoch 下降 → 学习是否收敛
- **右图**：把 `w·x + b = 0` 这条**直线**画在二维平面上，红色 = 预测 1，蓝色 = 预测 0

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.font_manager as fm
# 找出系统里真实存在的中文字体
CJK_CANDIDATES = ["PingFang SC", "Heiti SC", "Hiragino Sans GB",
                  "Songti SC", "STSong", "Arial Unicode MS",
                  "Microsoft YaHei", "SimHei", "WenQuanYi Zen Hei"]
available = {f.name for f in fm.fontManager.ttflist}
for cand in CJK_CANDIDATES:
    if cand in available:
        matplotlib.rcParams["font.sans-serif"] = [cand, "DejaVu Sans"]
        break
matplotlib.rcParams["axes.unicode_minus"] = False


def plot_training(model, X, y, title="Perceptron"):
    """画两个图：左 = 训练误差曲线，右 = 决策边界 + 样本点"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

    # ---------- 左图：每轮的错分类数 ----------
    ax1.plot(range(1, len(model.errors_history) + 1),
             model.errors_history, 'b-o', markersize=4)
    ax1.set_xlabel('Epoch（训练轮次）')
    ax1.set_ylabel('错分类样本数')
    ax1.set_title(f'{title}：训练误差曲线')
    ax1.set_ylim(-0.3, max(model.errors_history + [1]) + 0.5)
    ax1.grid(alpha=0.3)

    # ---------- 右图：决策边界 ----------
    x_min, x_max = -0.5, 1.5
    y_min, y_max = -0.5, 1.5
    xx1, xx2 = np.meshgrid(np.linspace(x_min, x_max, 200),
                            np.linspace(y_min, y_max, 200))
    grid = np.c_[xx1.ravel(), xx2.ravel()]
    Z = model.predict(grid).reshape(xx1.shape)

    # 背景：负类蓝、正类红
    ax2.contourf(xx1, xx2, Z, levels=[-0.1, 0.5, 1.1],
                 colors=['#cfe2ff', '#ffd6d6'], alpha=0.6)

    # 画决策直线 w·x + b = 0
    w, b = model.w, model.b
    if abs(w[1]) > 1e-6:
        x_line = np.array([x_min, x_max])
        y_line = -(w[0] * x_line + b) / w[1]
        ax2.plot(x_line, y_line, 'k-', linewidth=2, label='决策边界 w·x+b=0')
    elif abs(w[0]) > 1e-6:
        # 退化：w[1]≈0，直线是 x1 = -b/w[0] 的垂直线
        x_line = np.full(2, -b / w[0])
        ax2.plot(x_line, [y_min, y_max], 'k-', linewidth=2, label='决策边界')

    # 样本点
    for xi, yi in zip(X, y):
        color = 'red' if yi == 1 else 'blue'
        ax2.scatter(xi[0], xi[1], c=color, s=300,
                    edgecolors='k', zorder=3)

    ax2.set_xlim(x_min, x_max)
    ax2.set_ylim(y_min, y_max)
    ax2.set_xlabel('x1')
    ax2.set_ylabel('x2')
    ax2.set_title(f'{title}：决策边界  w={w.round(2)}, b={b:.2f}')
    ax2.legend(loc='upper left')
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


# ---------- 画 AND 训练结果 ----------
print("训练好的感知机参数：")
print(f"  w = {ppn.w}")
print(f"  b = {ppn.b:.3f}")
print(f"  预测 = {ppn.predict(X)}")
print(f"  真值 = {y_and}")
print()
plot_training(ppn, X, y_and, title="AND")

### ❌ 6. 感知机翻车现场：XOR

XOR 真值表：

| x1 | x2 | y |
|---|---|---|
| 0 | 0 | **0** |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | **0** |

红点（输出 1）在 (0,1) 和 (1,0)，蓝点（输出 0）在 (0,0) 和 (1,1)。
这四点**没有一条直线**能把红色和蓝色分开。**这就是"线性不可分"**。

In [ ]:
y_xor = np.array([0, 1, 1, 0])

# 训练 50 轮看看
ppn_xor = Perceptron(n_features=2, lr=0.1, seed=0)
ppn_xor.fit(X, y_xor, epochs=50, verbose=False)
plot_training(ppn_xor, X, y_xor, title="XOR (单层感知机)")

print("最终预测:", ppn_xor.predict(X))
print("真实标签:", y_xor)
print("→ 错误！感知机在 XOR 上根本不收敛（错误数在 1~2 之间反复横跳）。")

### 🏗️ 7. 多层感知机（MLP）：用「两层直线」拼出曲线

核心想法：
1. **第一层**画几条不同的直线，把空间切碎
2. 用 **sigmoid** 把直线变成柔和的概率（0~1）
3. **第二层**把这些"分碎的空间"再拼回去

网络结构：

```
x(2)  →  [隐藏层 h 个 sigmoid 神经元]  →  [输出层 1 个 sigmoid 神经元]  →  ŷ
```

> 关键替换：
> - 阶跃函数 → **sigmoid**（可微，才能用梯度下降）
> - 单层 → 两层

### 🔁 8. 反向传播：链式法则 + 一行一行倒着求梯度

设网络为：
- 隐藏层：`z1 = X @ W1 + b1`,  `a1 = sigmoid(z1)`
- 输出层：`z2 = a1 @ W2 + b2`,  `ŷ = sigmoid(z2)`
- 损失（二元交叉熵）：`L = -[ y·log(ŷ) + (1-y)·log(1-ŷ) ]`

链式法则（**对每个样本**算）：

```
dL/dz2 = ŷ - y                                    # 输出层误差
dL/dW2 = a1.T @ dL/dz2   / N
dL/db2 = mean(dL/dz2)

dL/da1 = dL/dz2 @ W2.T
dL/dz1 = dL/da1 * a1 * (1 - a1)                  # sigmoid 的导数
dL/dW1 = X.T @ dL/dz1      / N
dL/db1 = mean(dL/dz1)
```

> 看起来很多公式，但**本质就是「输出错了多少 → 每个权重分摊多少责任」**。

下面写代码。

### 💻 9. 手写一个两层 MLP

只用了 ~40 行核心代码：前向、反向、训练循环全部显式写出来。

In [ ]:
def sigmoid(z):
    # 为数值稳定用 np.clip
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def dsigmoid(a):
    """sigmoid 对 a 本身求导（a 已经是 sigmoid(z)）"""
    return a * (1.0 - a)


class MLP:
    """
    两层感知机：输入(2) -> 隐藏(h) -> 输出(1)
    激活: sigmoid + sigmoid
    损失: 二元交叉熵
    """
    def __init__(self, n_in, n_hidden=4, lr=0.5, seed=0):
        rng = np.random.default_rng(seed)
        # He 初始化 改写：sigmoid 用 Xavier 更稳
        self.W1 = rng.normal(0, 1/np.sqrt(n_in),   size=(n_in, n_hidden))
        self.b1 = np.zeros(n_hidden)
        self.W2 = rng.normal(0, 1/np.sqrt(n_hidden), size=(n_hidden, 1))
        self.b2 = np.zeros(1)
        self.lr = lr
        self.loss_history = []

    def forward(self, X):
        self.z1 = X @ self.W1 + self.b1            # (N, h)
        self.a1 = sigmoid(self.z1)                 # (N, h)
        self.z2 = self.a1 @ self.W2 + self.b2      # (N, 1)
        self.a2 = sigmoid(self.z2).ravel()         # (N,)
        return self.a2

    def backward(self, X, y, y_pred):
        N = len(X)
        y_pred = y_pred.reshape(-1, 1)             # (N, 1)

        # 输出层
        dz2 = (y_pred - y.reshape(-1, 1))          # (N, 1)  ← 交叉熵+sigmoid 的简化
        dW2 = (self.a1.T @ dz2) / N                # (h, 1)
        db2 = dz2.mean(axis=0)                     # (1,)

        # 隐藏层
        da1 = dz2 @ self.W2.T                      # (N, h)
        dz1 = da1 * dsigmoid(self.a1)              # (N, h)
        dW1 = (X.T @ dz1) / N                      # (2, h)
        db1 = dz1.mean(axis=0)                     # (h,)

        # 梯度下降
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2

    def fit(self, X, y, epochs=5000, verbose_every=1000):
        for ep in range(1, epochs + 1):
            y_pred = self.forward(X)
            # 交叉熵损失（加 1e-8 防止 log(0)）
            loss = -np.mean(y * np.log(y_pred + 1e-8) +
                            (1 - y) * np.log(1 - y_pred + 1e-8))
            self.loss_history.append(loss)
            self.backward(X, y, y_pred)
            if verbose_every and (ep % verbose_every == 0 or ep == 1):
                acc = (y_pred.round() == y).mean()
                print(f"epoch {ep:5d}  loss={loss:.4f}  acc={acc:.2f}")
        return self

    def predict(self, X, threshold=0.5):
        return (self.forward(X) >= threshold).astype(int)


# ---------- 在 XOR 上训练 ----------
mlp = MLP(n_in=2, n_hidden=4, lr=1.0, seed=1)
mlp.fit(X, y_xor, epochs=10000, verbose_every=2000)

print("\n最终预测概率:", mlp.forward(X).round(3))
print("最终预测类别:", mlp.predict(X))
print("真实标签    :", y_xor)

### 🎨 10. 看 MLP 怎么把 XOR 分开

不再是一条直线，而是 MLP 算出的「概率等高线」把红色和蓝色区域圈出来。

In [ ]:
def plot_decision_region(model, X, y, title="MLP", is_mlp=True):
    fig, ax = plt.subplots(figsize=(5.5, 5))

    # 建一个网格
    x1 = np.linspace(-0.5, 1.5, 200)
    x2 = np.linspace(-0.5, 1.5, 200)
    XX1, XX2 = np.meshgrid(x1, x2)
    grid = np.c_[XX1.ravel(), XX2.ravel()]

    if is_mlp:
        Z = model.forward(grid).reshape(XX1.shape)
    else:
        Z = model.predict(grid).reshape(XX1.shape)

    # 画背景色：红色=正类区，蓝色=负类区
    ax.contourf(XX1, XX2, Z, levels=[-0.1, 0.5, 1.1],
                colors=['#cfe2ff', '#ffd6d6'], alpha=0.6)
    # 等高线
    ax.contour(XX1, XX2, Z, levels=[0.5], colors='k', linewidths=2)

    # 样本点
    for xi, yi in zip(X, y):
        color = 'red' if yi == 1 else 'blue'
        ax.scatter(xi[0], xi[1], c=color, s=300, edgecolors='k', zorder=3)

    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel("x1"); ax.set_ylabel("x2")
    ax.set_title(f"{title}: 决策区域")
    ax.grid(alpha=0.3)
    plt.show()

# 单层感知机（直线分不开）
plot_decision_region(ppn_xor, X, y_xor, title="XOR (单层感知机)", is_mlp=False)

# 两层 MLP（曲线分开了）
plot_decision_region(mlp, X, y_xor, title="XOR (两层 MLP)", is_mlp=True)

### 📝 11. 一图回顾：感知机 vs MLP

| 维度 | 单层感知机 | 两层 MLP |
|---|---|---|
| 决策边界 | 一条**直线** | 多条直线拼出的**曲线** |
| 激活函数 | step（不可微） | sigmoid（可微） |
| 学习规则 | 感知机规则（错就改） | 梯度下降 + 反向传播 |
| 能解 | AND / OR / 任何**线性可分**问题 | **XOR** 等非线性问题 |
| 隐藏层 | 0 | 1+ |

### 🎓 你亲手实现的东西

- ✅ `Perceptron` 类：前向 + 感知机学习规则
- ✅ `MLP` 类：两层网络 + sigmoid + **手写反向传播**
- ✅ 训练可视化：误差曲线、决策边界、决策区域

### 🚀 下一步可以学什么？

1. **换激活函数**：把 sigmoid 换成 ReLU（梯度不消失，训练更深网络）
2. **加更多层**：3 层、4 层... 但要小心梯度消失 / 爆炸
3. **换损失函数**：多分类用 softmax + 交叉熵
4. **加正则化**：Dropout、L2，防止过拟合
5. **用 mini-batch / SGD**：不再逐样本更新
6. **试真实数据**：MNIST 手写数字分类

> 之后可以读 PyTorch / TensorFlow 源码——你会发现「卷积」「注意力」「归一化」全都是这套前向+反向+梯度下降的扩展。

🌟 **恭喜你完成了第一个「从零开始的神经网络」！** 改一改隐藏神经元数量、学习率、跑几遍，看 MLP 在 XOR 上是不是也能翻车。